# Notebook to inspect and save lapresse database

To run, you need to download [initial_datasets_NOT_REQUIRED](https://drive.google.com/drive/u/0/folders/1dRD_6FZdgt7GEkoqSJq_BTi_AjITnHq5) and places both files in `/data/`

In [ ]:
import sqlite3
import pandas as pd
import os

In [ ]:
conn = sqlite3.connect('../../data/lapresse.db')
cursor = conn.cursor()

In [ ]:
# Fetch all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [row[0] for row in cursor.fetchall()]

print(f"\nDatabase contains {len(tables)} tables:\n")

for table in tables:
    print(f"Table: {table}")
    cursor.execute(f"PRAGMA table_info({table});")
    columns = cursor.fetchall()
    
    for col in columns:
        col_name, col_type = col[1], col[2]
        print(f"    - {col_name} ({col_type})")
    print()

In [ ]:
df = pd.read_sql_query("SELECT * FROM entryversion", conn)

print("Basic statistics - ALL versions:")

total_versions = len(df)
unique_articles = df['entry_id'].nunique()

df['word_count'] = df['summary'].apply(lambda x: len(str(x).split()))

print(f"Total article versions: {total_versions}")
print(f"Unique articles (entry_id): {unique_articles}")
print(f"Average word count per summary: {df['word_count'].mean():.2f}")
print(f"Min word count: {df['word_count'].min()}")
print(f"Max word count: {df['word_count'].max()}")

print("\n")

print("Basic statistics - NEWEST version per article:")

# Sort by version number (higher version = newer) and keep only the latest per entry_id
df_latest = df.sort_values(['entry_id', 'version'], ascending=[True, False]).drop_duplicates(subset=['entry_id'], keep='first')

total_latest = len(df_latest)
print(f"Total articles (only newest version): {total_latest}")
print(f"Average word count per summary (newest versions only): {df_latest['word_count'].mean():.2f}")
print(f"Min word count (newest versions only): {df_latest['word_count'].min()}")
print(f"Max word count (newest versions only): {df_latest['word_count'].max()}")

In [ ]:
df_top6000 = df_latest.sort_values('word_count', ascending=False).head(6000)

print("Basic statistics - Top 6000 longest articles (newest versions only):")

total_articles = len(df_top6000)
average_word_count = df_top6000['word_count'].mean()
min_word_count = df_top6000['word_count'].min()
max_word_count = df_top6000['word_count'].max()

print(f"Average word count: {average_word_count:.2f}")
print(f"Min word count: {min_word_count}")
print(f"Max word count: {max_word_count}")

In [ ]:
longest_article = df_top6000.iloc[0]

print(f"\nArticle with the most words (word count = {longest_article['word_count']}):\n")
print(longest_article['summary'])

In [ ]:
import os
import pandas as pd
import re
import shutil
import string
import html

save_dir = "../../data/raw_lapresse_dataset/"

# Clear the folder before saving new files
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)  # Remove all existing files and subdirectories
os.makedirs(save_dir, exist_ok=True)  # Recreate the empty directory

# Function to clean filenames
def clean_filename(title):
    title = title.strip() 
    title = re.sub(r'[\/:*?"<>|]', '', title) 
    title = re.sub(r'\s+', ' ', title) 
    return title[:150].strip()

for _, row in df_top6000.iterrows():
    title = row['title']  
    summary = row['summary'] 

    # Fix encoding issues
    summary = ''.join(c for c in summary if c in string.printable)  # Remove invisible characters
    summary = summary.replace("\r\n", "\n").replace("\r", "\n") 
    summary = html.unescape(summary)  # Convert HTML entities (e.g., &#13; -> newline)

    # Create a valid filename
    filename = f"{clean_filename(title)}.txt"
    file_path = os.path.join(save_dir, filename)
    
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(summary)

print(f"Cleared the folder and saved {len(df_top6000)} files in {save_dir}")